In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


Net = nn.Sequential(nn.Linear(20,4), nn.ReLU(), nn.Linear(4,1))
X = torch.rand(64,20)
Net(X)
print(Net[2].state_dict())

print(Net[2].bias.grad)

print(Net)

OrderedDict([('weight', tensor([[-0.2567, -0.0559, -0.0215, -0.1569]])), ('bias', tensor([0.3727]))])
None
Sequential(
  (0): Linear(in_features=20, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=1, bias=True)
)


In [2]:
print(Net[2].weight)
print()
print(Net[2].weight.grad)

Parameter containing:
tensor([[-0.2567, -0.0559, -0.0215, -0.1569]], requires_grad=True)

None


每个参数都表示为参数类的一个实例，是复合的对象：包含值、梯度和额外信息。

In [3]:
for name, param in Net.named_parameters():
    print(f"参数名字: {name}, 形状: {param.shape}")
    print(param)

参数名字: 0.weight, 形状: torch.Size([4, 20])
Parameter containing:
tensor([[ 0.0934,  0.1751, -0.0839, -0.0316, -0.0681,  0.0588, -0.0229,  0.1105,
          0.1994,  0.1451, -0.0516,  0.2111, -0.1965, -0.1043,  0.1285,  0.0872,
         -0.0357,  0.2214,  0.1932,  0.1481],
        [ 0.0351,  0.0421, -0.2096,  0.1868, -0.0766, -0.1323, -0.1187,  0.1297,
         -0.1238, -0.0361, -0.2109, -0.2073, -0.0025, -0.0271,  0.0562,  0.1322,
         -0.1808, -0.1059, -0.0997, -0.2185],
        [-0.0343, -0.1371, -0.0258,  0.0329,  0.1985,  0.0390,  0.0176,  0.0135,
         -0.0625, -0.2195, -0.1552,  0.1731,  0.1369, -0.1550,  0.1704,  0.1395,
          0.0384, -0.1810,  0.0821,  0.0181],
        [ 0.1101, -0.0176, -0.2212, -0.0763,  0.0707,  0.2046,  0.0683, -0.0297,
         -0.0840, -0.2004,  0.0398,  0.0583, -0.0146,  0.1328, -0.1988, -0.0781,
          0.0208, -0.0382, -0.0894, -0.2111]], requires_grad=True)
参数名字: 0.bias, 形状: torch.Size([4])
Parameter containing:
tensor([0.0915, 0.2056, 0.148

In [4]:
def block1():
    return nn.Sequential(nn.Linear(2,4), nn.ReLU(), nn.Linear(4,2), nn.ReLU())


def block2():
    Net = nn.Sequential()
    for i in range(4):
        Net.add_module(f'block{i}',block1())
    return Net

BigNet = nn.Sequential(block2(), nn.Linear(2,1))
print(BigNet)
# print(BigNet[0])

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=2, out_features=4, bias=True)
      (1): ReLU()
      (2): Linear(in_features=4, out_features=2, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=2, out_features=1, bias=True)
)


In [5]:
# print(BigNet[0])
print(BigNet[1])

Linear(in_features=2, out_features=1, bias=True)


In [6]:
print(BigNet[0][2])

Sequential(
  (0): Linear(in_features=2, out_features=4, bias=True)
  (1): ReLU()
  (2): Linear(in_features=4, out_features=2, bias=True)
  (3): ReLU()
)


In [7]:
print(BigNet[0][2][2].weight)

Parameter containing:
tensor([[ 0.3238, -0.4085, -0.0431,  0.2619],
        [ 0.3044, -0.1383,  0.2897,  0.1111]], requires_grad=True)


In [8]:
def init_params(block):
    if type(block) == nn.Linear:
        nn.init.normal_(block.weight, 0, 0.01)
        nn.init.normal_(block.bias, 0, 0.01)

BigNet.apply(init_params)
BigNet[0][0][0].weight.data[0], BigNet[0][0][0].bias.data[0]

(tensor([0.0035, 0.0119]), tensor(0.0093))

### torch.nn.init
专门用于初始化参数的，
nn.init.uniform_(w, a=0, b=1)：均匀分布。
nn.init.zeros_(w) / nn.init.ones_(w)：全 0 或 全 1（通常用来初始化偏置 bias）。

针对特定激活函数设计的初始化：

nn.init.xavier_uniform_(w) / nn.init.xavier_normal_(w)： 也叫 Glorot 初始化。适合搭配 Sigmoid 或 Tanh 激活函数的网络，能有效缓解梯度消失。

nn.init.kaiming_uniform_(w) / nn.init.kaiming_normal_(w)： 也叫 He 初始化。专门为 ReLU 及其变体激活函数量身定制的初始化方法。

nn.init里的函数名最后都有一个下划线 _
在 PyTorch 中，结尾带 _ 的函数代表原地操作 (In-place)。意思是：它不创建新的内存空间，直接把传入的张量内部的数值给强行改写掉。


### 默认初始化
每一个内置的层都有一个隐藏的函数叫做 reset_parameters()。当你执行类似 self.hidden = nn.Linear(20, 256) 时，PyTorch 的 __init__ 函数最后一步就会自动调用reset_parameters()。
默认初始化使用了一种根据输入维度自适应的均匀分布。假设输入维度是 N，权重、偏置会被初始化为一个均匀分布。分布的范围是：$$\left[ -\frac{1}{\sqrt{N_{in}}}, \frac{1}{\sqrt{N_{in}}} \right]$$
目的是为了控制方差，防止数值爆炸或消失。输入连接的神经元越多（N越大），初始化的权重就应该越小。


### apply：PyTorch中的声明式编程
只需要定义对单个层要做什么，然后 .apply，将自动遍历。

In [9]:
def init_xavier(block):
    if type(block) == nn.Linear:
        nn.init.xavier_normal_(block.weight)

def init_42(block):
    if type(block) == nn.Linear:
        nn.init.constant_(block.weight, 42)

BigNet[0][0][0].apply(init_xavier)
BigNet[0][0][2].apply(init_42)

print(BigNet[0][0][0].weight)
print(BigNet[0][0][2].weight)

Parameter containing:
tensor([[-0.1661,  0.2824],
        [-0.2472,  0.6741],
        [ 0.5080,  0.1235],
        [-0.3718,  0.9716]], requires_grad=True)
Parameter containing:
tensor([[42., 42., 42., 42.],
        [42., 42., 42., 42.]], requires_grad=True)


In [10]:
BigNet[0][0][0].weight + 1
BigNet[0][0][0].weight.data[0][0] = 99
print(BigNet[0][0][0].weight)

Parameter containing:
tensor([[99.0000,  0.2824],
        [-0.2472,  0.6741],
        [ 0.5080,  0.1235],
        [-0.3718,  0.9716]], requires_grad=True)


### 共享参数
参数共享在CV，NLP，注意力等领域有很大作用，共享的参数的梯度最终会求和在一起

In [11]:
share_block = nn.Linear(3,8)
net = nn.Sequential(nn.Linear(10,3), nn.ReLU(), share_block, nn.ReLU(), nn.Linear(8,3), nn.ReLU(), share_block, nn.ReLU(), nn.Linear(8,2))

print(net[2].weight == net[6].weight)

tensor([[True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True]])
